# Finite differences

_______

## 1) Function definition

We want to approximate the derivative of $f : x \mapsto \arctan(x) + 3 \cos^2(2x) + 7 x$

In [ ]:
import numpy as np

def f(x):
    """ Function to derivate """
    return np.atan(x) + 3 * np.cos(2*x)**2 +7 * x

def df_dx(x):
    """ Reference derivative """
    return 1/(x**2 + 1) - 12*np.cos(2*x)*np.sin(2*x) + 7


xref = np.random.rand(1)
dfdx_ref = df_dx(xref)

print(f"df_dx(xref) = {dfdx_ref}")

## 2) Finite differences

If $f$ is regular enough we can check numerically that 
$$ \frac{df}{dx} = \frac{f(x+h)-f(x)}{h} + O(h) $$
$$ \frac{df}{dx} = \frac{f(x+h/2)-f(x-h/2)}{h} + O(h^2) $$

In [ ]:
def FD1(f, x, h):
    return ( f(x+h)-f(x) )/ h

def FD2(f, x, h):
    return ( f(x+h/2)-f(x-h/2) ) / h

In [ ]:
H = np.logspace(-15, -1, 50)

errorFD1 = [abs(FD1(f, xref, h) - dfdx_ref)  for h in H]
errorFD2 = [abs(FD2(f, xref, h) - dfdx_ref)  for h in H]

import matplotlib.pyplot as plt

plt.loglog(H, errorFD1, 'o', label = "FD1")
plt.loglog(H, errorFD2, '*', label = "FD2")
plt.loglog(H, H, '--', label = "$h$")
plt.loglog(H, H**2, ':', label = "$h^2$")
plt.xlabel("h"); plt.ylabel("norm of the error"); plt.title("Error on the derivative approximation")
plt.legend(loc = 'best'); plt.grid(); plt.show()

The precision of the derivative approximation is limited by floating point computation, due to the substraction of two "big" quantities. It's a **catastrophic cancellation** issue (sic).

Can we overcome this?

## 3) Complex step

**Reference** : Squire, William, and Trapp, George, Using complex variables to estimate derivatives of real functions, SIAM Review 40, 1998, pp. 110-112. epubs.siam.org/doi/abs/10.1137/S003614459631241X

If $f$ is regular enough and analytic, a Taylor expansion gives

$$f(x+ih)=f(x)+ i h \frac{df}{dx}(x)− \frac{h^2}{2!} \frac{d^2f}{dx^2}(x) − \frac{ih^3}{3!} \frac{d^3f}{dx^3}(x) +...$$

$$\frac{f(x+ih)-f(x)}{h} = i \frac{df}{dx}(x) − \frac{h}{2!} \frac{d^2f}{dx^2}(x) − \frac{ih^2}{3!} \frac{d^3f}{dx^3}(x) +...$$

$$\Rightarrow   \boxed{\text{Im} \left(\frac{f(x+ih)-f(x)}{h} \right) = \frac{df}{dx}(x) + O(h^2)} $$

But this time, we have **no cancellation error**, because there is **no substraction in the imaginary part** ! We can check it numerically.



In [ ]:
def cplxStep(f, x, h):
    return (( f(x+ 1j*h)-f(x) )/ h).imag

In [ ]:
errorCplxStep = [abs(cplxStep(f, xref, h) - dfdx_ref)  for h in H]

import matplotlib.pyplot as plt

plt.loglog(H, errorFD1, 'o', label = "FD1")
plt.loglog(H, errorFD2, '*', label = "FD2")
plt.loglog(H, errorCplxStep, '+', label = "cplx step")
plt.loglog(H, H, '--', label = "$h$")
plt.loglog(H, H**2, ':', label = "$h^2$")
plt.xlabel("h"); plt.ylabel("norm of the error"); plt.title("Error on the derivative approximation")
plt.legend(loc = 'best'); plt.grid(); plt.show()

We can reach machine precision (about 15 to 17 digits), with $h$ as small as the smallest possible float.


In [ ]:
Hextreme = np.logspace(-300, -1, 1000)
errorCplxStep = [abs(cplxStep(f, xref, h) - dfdx_ref)  for h in Hextreme]

import matplotlib.pyplot as plt

plt.loglog(Hextreme, errorCplxStep, '+', label = "cplx step")
plt.xlabel("h"); plt.ylabel("norm of the error"); plt.title("Error on the derivative approximation")
plt.legend(loc = 'best'); plt.grid(); plt.show()